In [2]:
# ============================================================
# IMPORTAÇÕES
# ============================================================

import time
import itertools
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    precision_recall_curve,
    auc,
    log_loss
)


# ============================================================
# FUNÇÃO PARA TRUNCAR EM 6 CASAS DECIMAIS
# ============================================================

def truncar_6(x):
    """
    Trunca um número em 6 casas decimais, sem arredondar.
    """
    return np.trunc(float(x) * 1_000_000) / 1_000_000


# ============================================================
# LEITURA DO CSV
# ============================================================

df = pd.read_csv("creditcard.csv")


# ============================================================
# DEFINIÇÃO DO TARGET
# ============================================================

if "status_fraude" in df.columns:
    target_name = "status_fraude"

elif "Class" in df.columns:
    df = df.rename(columns={"Class": "status_fraude"})
    target_name = "status_fraude"

else:
    raise ValueError("Não encontrei a coluna target: 'status_fraude' ou 'Class'.")


# ============================================================
# SELEÇÃO DAS FEATURES NUMÉRICAS
# ============================================================

features = [
    col for col in df.columns
    if col != target_name
    and pd.api.types.is_numeric_dtype(df[col])
]


# ============================================================
# COMBINAÇÕES 2x2
# ============================================================

combinacoes_2x2 = list(itertools.combinations(features, 2))

print("Dataset carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")
print(f"Target utilizado: {target_name}")
print(f"Quantidade de features numéricas: {len(features)}")
print(f"Quantidade de combinações 2x2: {len(combinacoes_2x2)}")


# ============================================================
# LOG-PDF GAUSSIANA MULTIVARIADA
# ============================================================

def logpdf_gaussiana_multivariada(X, media, cov):
    """
    Calcula log N(x | media, cov).

    Funciona para 1D e múltiplas dimensões.
    No caso 2x2, X terá duas colunas.
    """

    X = np.asarray(X)
    media = np.asarray(media)
    cov = np.asarray(cov)

    if X.ndim == 1:
        X = X.reshape(-1, 1)

    if media.ndim == 0:
        media = np.array([media])

    media = media.reshape(-1)

    cov = np.atleast_2d(cov)

    n_features = X.shape[1]

    sinal, logdet = np.linalg.slogdet(cov)

    if sinal <= 0:
        return np.full(X.shape[0], -np.inf)

    diff = X - media

    solucao = np.linalg.solve(cov, diff.T).T

    termo_quadratico = np.sum(
        diff * solucao,
        axis=1
    )

    logpdf = -0.5 * (
        n_features * np.log(2 * np.pi)
        + logdet
        + termo_quadratico
    )

    return logpdf


# ============================================================
# LOG-VEROSSIMILHANÇA COM RÓTULO
# ============================================================

def calcular_log_veross_com_rotulo(
    X_scaled,
    y_real,
    reg_covar=1e-6
):
    """
    Calcula a log-verossimilhança média por amostra usando o rótulo real.

    Ideia:
    - y = 0 define uma Gaussiana para não fraude
    - y = 1 define uma Gaussiana para fraude
    - os pesos são as proporções reais das classes

    Retorna:
    - log-verossimilhança média por amostra.
    """

    X_scaled = np.asarray(X_scaled)

    if X_scaled.ndim == 1:
        X_scaled = X_scaled.reshape(-1, 1)

    y_real = np.asarray(y_real).astype(int)

    n_amostras, n_features = X_scaled.shape

    log_veross_total = 0.0

    for classe in [0, 1]:

        X_classe = X_scaled[y_real == classe]

        n_classe = X_classe.shape[0]

        if n_classe <= 1:
            return np.nan

        peso_classe = n_classe / n_amostras

        media_classe = np.mean(
            X_classe,
            axis=0
        )

        cov_classe = np.cov(
            X_classe,
            rowvar=False
        )

        cov_classe = np.atleast_2d(cov_classe)

        cov_classe = cov_classe + reg_covar * np.eye(n_features)

        logpdf_classe = logpdf_gaussiana_multivariada(
            X=X_classe,
            media=media_classe,
            cov=cov_classe
        )

        log_veross_total += np.sum(
            np.log(peso_classe) + logpdf_classe
        )

    log_veross_media = log_veross_total / n_amostras

    return log_veross_media


# ============================================================
# FUNÇÃO OTIMIZADA PARA ENCONTRAR O MELHOR PONTO DE CORTE
# PELO MCC USANDO AS PRÓPRIAS PROBABILIDADES COMO THRESHOLDS
# ============================================================

def encontrar_melhor_ponto_corte_mcc(y_real, probabilidades):
    """
    Encontra o melhor ponto de corte pelo MCC.

    Usa como thresholds as próprias probabilidades estimadas pelo modelo,
    mas calcula tudo de forma otimizada via ordenação e somas acumuladas.

    Regra:
        y_pred = 1 se probabilidade >= threshold
        y_pred = 0 caso contrário
    """

    y_real = np.asarray(y_real).astype(int)
    probabilidades = np.asarray(probabilidades)

    ordem = np.argsort(-probabilidades)

    probs_ord = probabilidades[ordem]
    y_ord = y_real[ordem]

    total_positivos = np.sum(y_ord == 1)
    total_negativos = np.sum(y_ord == 0)

    tp_acum = np.cumsum(y_ord == 1)
    fp_acum = np.cumsum(y_ord == 0)

    fn_acum = total_positivos - tp_acum
    tn_acum = total_negativos - fp_acum

    numerador = (tp_acum * tn_acum) - (fp_acum * fn_acum)

    denominador = np.sqrt(
        (tp_acum + fp_acum) *
        (tp_acum + fn_acum) *
        (tn_acum + fp_acum) *
        (tn_acum + fn_acum)
    )

    mccs = np.divide(
        numerador,
        denominador,
        out=np.zeros_like(numerador, dtype=float),
        where=denominador != 0
    )

    indices_validos = np.r_[
        np.where(probs_ord[:-1] != probs_ord[1:])[0],
        len(probs_ord) - 1
    ]

    mccs_validos = mccs[indices_validos]

    melhor_idx_local = np.argmax(mccs_validos)
    melhor_idx = indices_validos[melhor_idx_local]

    melhor_ponto_corte = probs_ord[melhor_idx]
    melhor_mcc = mccs[melhor_idx]

    return melhor_ponto_corte, melhor_mcc


# ============================================================
# FUNÇÃO PARA ANALISAR UMA COMBINAÇÃO 2x2
# ============================================================

def analisar_combinacao_2x2(
    df,
    feature_1,
    feature_2,
    target_name="status_fraude",
    resumo_combinacoes=None,
    verbose=False
):

    if resumo_combinacoes is None:
        resumo_combinacoes = []

    inicio = time.perf_counter()

    if verbose:
        print(f"\nIniciando combinação: {feature_1} + {feature_2}")

    # ========================================================
    # DADOS
    # ========================================================

    temp = df[[feature_1, feature_2, target_name]].dropna()

    if temp.empty:
        if verbose:
            print("  - Ignorada: dados vazios após dropna")
        return resumo_combinacoes

    X = temp[[feature_1, feature_2]]
    y_real = temp[target_name].astype(int)

    if y_real.nunique() < 2:
        if verbose:
            print("  - Ignorada: target possui apenas uma classe")
        return resumo_combinacoes

    if X[feature_1].nunique() < 2 or X[feature_2].nunique() < 2:
        if verbose:
            print("  - Ignorada: uma das features é constante")
        return resumo_combinacoes

    # ========================================================
    # ESCALONAMENTO
    # ========================================================

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # ========================================================
    # TREINAMENTO GMM 2D
    # ========================================================

    try:
        gmm = GaussianMixture(
            n_components=2,
            covariance_type="full",
            random_state=42,
            n_init=3,
            reg_covar=1e-6
        )

        gmm.fit(X_scaled)

    except Exception as erro:
        if verbose:
            print(f"  - Erro no treinamento da GMM: {erro}")
        return resumo_combinacoes

    # ========================================================
    # LOG-VEROSSIMILHANÇA DA GMM SEM RÓTULO
    # ========================================================
    # gmm.score(X_scaled) retorna a log-verossimilhança média
    # por amostra segundo a GMM ajustada sem usar o rótulo.

    log_veross_gmm = gmm.score(X_scaled)

    # ========================================================
    # LOG-VEROSSIMILHANÇA COM RÓTULO
    # ========================================================
    # Aqui o rótulo real define as duas Gaussianas:
    # uma para y=0 e outra para y=1.

    log_veross_com_rotulo = calcular_log_veross_com_rotulo(
        X_scaled=X_scaled,
        y_real=y_real,
        reg_covar=1e-6
    )

    # ========================================================
    # NEGATIVE LOG-LIKELIHOODS
    # ========================================================
    # Agora trabalhamos com -log-verossimilhança.
    # Nesse caso, menor é melhor.

    neg_log_veross_com_rotulo = -log_veross_com_rotulo
    neg_log_veross_gmm = -log_veross_gmm

    diferenca_neg_log_veross = (
        neg_log_veross_com_rotulo
        - neg_log_veross_gmm
    )

    # ========================================================
    # IDENTIFICAR CLUSTER ASSOCIADO À FRAUDE
    # ========================================================

    clusters = gmm.predict(X_scaled)

    ct = pd.crosstab(
        clusters,
        y_real
    )

    if 1 not in ct.columns:
        if verbose:
            print("  - Ignorada: classe 1 não encontrada no crosstab")
        return resumo_combinacoes

    cluster_fraude = ct[1].idxmax()

    # ========================================================
    # PROBABILIDADES DO CLUSTER ASSOCIADO À FRAUDE
    # ========================================================

    probabilidades = gmm.predict_proba(X_scaled)[:, cluster_fraude]

    probabilidades = np.clip(
        probabilidades,
        1e-15,
        1 - 1e-15
    )

    # ========================================================
    # AUC-PR
    # ========================================================

    precision_vals, recall_vals, _ = precision_recall_curve(
        y_real,
        probabilidades
    )

    auc_pr = auc(
        recall_vals,
        precision_vals
    )

    # ========================================================
    # MELHOR PONTO DE CORTE E MCC
    # ========================================================

    melhor_ponto_corte, mcc = encontrar_melhor_ponto_corte_mcc(
        y_real=y_real,
        probabilidades=probabilidades
    )

    # ========================================================
    # PONTO DE CORTE MÉDIO
    # ========================================================

    ponto_corte_medio = 0.5

    # ========================================================
    # LOG LOSS
    # ========================================================

    ll = log_loss(
        y_real,
        probabilidades
    )

    fim = time.perf_counter()

    # ========================================================
    # NORMALIZAÇÕES
    # ========================================================

    auc_pr_norm = np.clip(
        auc_pr,
        0,
        1
    )

    mcc_norm = (mcc + 1) / 2

    mcc_norm = np.clip(
        mcc_norm,
        0,
        1
    )

    log_loss_norm = 1 / (1 + ll)

    log_loss_norm = np.clip(
        log_loss_norm,
        0,
        1
    )

    # ========================================================
    # SCORE FINAL
    # ========================================================

    score_final = np.mean([
        auc_pr_norm,
        mcc_norm,
        log_loss_norm
    ])

    # ========================================================
    # RESULTADO COM TRUNCAMENTO EM 6 CASAS
    # ========================================================

    nova_linha = {
        "Feature_1": feature_1,
        "Feature_2": feature_2,
        "Combinacao": f"{feature_1} + {feature_2}",

        "AUC_PR": truncar_6(float(auc_pr)),
        "MCC": truncar_6(float(mcc)),
        "Log_Loss": truncar_6(float(ll)),
        "Log_Loss_Norm": truncar_6(float(log_loss_norm)),

        "Neg_Log_Veross_Com_Rotulo": truncar_6(float(neg_log_veross_com_rotulo)),
        "Neg_Log_Veross_GMM": truncar_6(float(neg_log_veross_gmm)),
        "Diferenca_Neg_Log_Veross": truncar_6(float(diferenca_neg_log_veross)),

        "Score_Final": truncar_6(float(score_final)),
        "Melhor_Ponto_Corte": truncar_6(float(melhor_ponto_corte)),
        "Ponto_Corte_Medio": truncar_6(float(ponto_corte_medio)),
        "Tempo": truncar_6(float(fim - inicio))
    }

    resumo_combinacoes.append(nova_linha)

    if verbose:
        print(f"  - Finalizada em {fim - inicio:.2f} segundos")
        print(f"  - Score_Final: {score_final:.6f}")
        print(f"  - Neg_Log_Veross_Com_Rotulo: {neg_log_veross_com_rotulo:.6f}")
        print(f"  - Neg_Log_Veross_GMM: {neg_log_veross_gmm:.6f}")
        print(f"  - Diferenca_Neg_Log_Veross: {diferenca_neg_log_veross:.6f}")

    return resumo_combinacoes


# ============================================================
# EXECUÇÃO PARA TODAS AS COMBINAÇÕES 2x2
# ============================================================

resumo_combinacoes = []

inicio_geral = time.perf_counter()

for feature_1, feature_2 in tqdm(
    combinacoes_2x2,
    desc="Processando combinações 2x2",
    unit="combinação"
):
    tamanho_antes = len(resumo_combinacoes)

    resumo_combinacoes = analisar_combinacao_2x2(
        df=df,
        feature_1=feature_1,
        feature_2=feature_2,
        target_name=target_name,
        resumo_combinacoes=resumo_combinacoes,
        verbose=False
    )

    tamanho_depois = len(resumo_combinacoes)

    if tamanho_depois > tamanho_antes:
        tqdm.write(f"Combinação processada: {feature_1} + {feature_2}")
    else:
        tqdm.write(f"Combinação ignorada ou com erro: {feature_1} + {feature_2}")

fim_geral = time.perf_counter()


# ============================================================
# DATAFRAME FINAL
# ============================================================

scores_2x2 = pd.DataFrame(resumo_combinacoes)

if scores_2x2.empty:
    raise ValueError(
        "Nenhuma combinação 2x2 foi processada. "
        "Verifique se existem features numéricas válidas e se o target está correto."
    )

scores_2x2 = scores_2x2.sort_values(
    by="Score_Final",
    ascending=False
).reset_index(drop=True)

scores_2x2["Posicao_Rank"] = np.arange(
    1,
    len(scores_2x2) + 1
)

scores_2x2 = scores_2x2[
    [
        "Feature_1",
        "Feature_2",
        "Combinacao",
        "AUC_PR",
        "MCC",
        "Log_Loss",
        "Log_Loss_Norm",
        "Neg_Log_Veross_Com_Rotulo",
        "Neg_Log_Veross_GMM",
        "Diferenca_Neg_Log_Veross",
        "Score_Final",
        "Melhor_Ponto_Corte",
        "Ponto_Corte_Medio",
        "Tempo",
        "Posicao_Rank"
    ]
]


# ============================================================
# EXPORTAÇÃO PARA CSV
# ============================================================

scores_2x2.to_csv(
    "2x2_scores.csv",
    index=False,
    encoding="utf-8-sig",
    float_format="%.6f"
)


# ============================================================
# RELATÓRIO FINAL
# ============================================================

tempo_total_segundos = fim_geral - inicio_geral
tempo_total_minutos = tempo_total_segundos / 60

print("\nProcessamento finalizado.")
print(f"Combinações processadas com sucesso: {len(scores_2x2)}")
print(f"Tempo total: {tempo_total_segundos:.2f} segundos")
print(f"Tempo total: {tempo_total_minutos:.2f} minutos")
print("Arquivo salvo como: 2x2_scores.csv")

print("\nTop 20 combinações:")
display(scores_2x2.head(20))

Dataset carregado com sucesso.
Shape do dataset: (283726, 31)
Target utilizado: status_fraude
Quantidade de features numéricas: 30
Quantidade de combinações 2x2: 435


Processando combinações 2x2:   0%|          | 0/435 [00:00<?, ?combinação/s]

Combinação processada: tempo_desde_a_primeira_transacao + V1
Combinação processada: tempo_desde_a_primeira_transacao + V2
Combinação processada: tempo_desde_a_primeira_transacao + V3
Combinação processada: tempo_desde_a_primeira_transacao + V4
Combinação processada: tempo_desde_a_primeira_transacao + V5
Combinação processada: tempo_desde_a_primeira_transacao + V6
Combinação processada: tempo_desde_a_primeira_transacao + V7
Combinação processada: tempo_desde_a_primeira_transacao + V8
Combinação processada: tempo_desde_a_primeira_transacao + V9
Combinação processada: tempo_desde_a_primeira_transacao + V10
Combinação processada: tempo_desde_a_primeira_transacao + V11
Combinação processada: tempo_desde_a_primeira_transacao + V12
Combinação processada: tempo_desde_a_primeira_transacao + V13
Combinação processada: tempo_desde_a_primeira_transacao + V14
Combinação processada: tempo_desde_a_primeira_transacao + V15
Combinação processada: tempo_desde_a_primeira_transacao + V16
Combinação proces

,Feature_1,Feature_2,Combinacao,AUC_PR,MCC,Log_Loss,Log_Loss_Norm,Neg_Log_Veross_Com_Rotulo,Neg_Log_Veross_GMM,Diferenca_Neg_Log_Veross,Score_Final,Melhor_Ponto_Corte,Ponto_Corte_Medio,Tempo,Posicao_Rank
0,V14,V17,V14 + V17,0.639355,0.656532,0.263305,0.791574,2.660656,2.490707,0.169948,0.753065,0.999999,0.5,8.341958,1
1,V11,V17,V11 + V17,0.559834,0.593638,0.126620,0.887610,2.716309,2.642419,0.073889,0.748088,0.999999,0.5,16.744053,2
2,V17,V22,V17 + V22,0.500903,0.547170,0.085365,0.921348,2.735002,2.660432,0.074570,0.731945,0.999999,0.5,13.056883,3
3,V17,V26,V17 + V26,0.511800,0.559867,0.158952,0.862847,2.735615,2.668668,0.066946,0.718194,0.999999,0.5,17.104448,4
4,V13,V17,V13 + V17,0.508750,0.555861,0.165123,0.858278,2.735535,2.671020,0.064514,0.714986,0.999999,0.5,16.446156,5
5,V15,V17,V15 + V17,0.506026,0.552769,0.165547,0.857965,2.735589,2.668123,0.067466,0.713459,0.999999,0.5,15.467909,6
6,V14,V16,V14 + V16,0.623919,0.644108,0.494488,0.669125,2.753917,2.685687,0.068230,0.705033,0.999999,0.5,10.957835,7
7,V12,V16,V12 + V16,0.658446,0.708266,0.662315,0.601570,2.765243,2.674933,0.090309,0.704716,0.999999,0.5,7.884551,8
8,V16,V17,V16 + V17,0.532500,0.574765,0.278810,0.781976,2.691257,2.578374,0.112882,0.700620,0.999999,0.5,13.861935,9
9,V4,V17,V4 + V17,0.539237,0.558570,0.279822,0.781358,2.722954,2.632157,0.090797,0.699960,0.999999,0.5,11.208336,10
